In [18]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split


In [19]:
# ====== Загрузка данных ======
df = pd.read_csv("train_valid.csv")


In [20]:
# ==================== ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ ====================
# Создаем копию данных
df_processed_train = df.copy()

# НОВЫЕ ПРИЗНАКИ: Взаимодействия (добавляем в самом начале)
df_processed_train['Rooms_Square_Ratio'] = df_processed_train['Rooms'] / df_processed_train['Square']
df_processed_train['LifeSquare_Ratio'] = df_processed_train['LifeSquare'] / df_processed_train['Square']

# На тренировочных данных вычисляем средние
target_encoding = df_processed_train.groupby('DistrictId')['Price'].mean()
price_per_sqm_by_district = df_processed_train.groupby('DistrictId')['Price'].mean() / df_processed_train.groupby('DistrictId')['Square'].mean()
room_means = df_processed_train.groupby(['DistrictId', 'Rooms'])['Price'].mean().reset_index()
room_means.rename(columns={'Price': 'Mean_Price_By_Rooms'}, inplace=True)

# НОВЫЙ ПРИЗНАК: Средняя цена за кв.м жилой площади по району и количеству комнат
life_square_means = df_processed_train.groupby(['DistrictId', 'Rooms']).apply(
    lambda x: x['Price'].mean() / x['LifeSquare'].mean() if x['LifeSquare'].mean() > 0 else 0
).reset_index()
life_square_means.rename(columns={0: 'Mean_Price_By_LifeSquare'}, inplace=True)

# НОВЫЙ ПРИЗНАК: Средняя цена за кв.м общей площади по району и количеству комнат
square_means = df_processed_train.groupby(['DistrictId', 'Rooms']).apply(
    lambda x: x['Price'].mean() / x['Square'].mean() if x['Square'].mean() > 0 else 0
).reset_index()
square_means.rename(columns={0: 'Mean_Price_By_Square'}, inplace=True)

train_target_encoding = target_encoding
train_price_per_sqm = price_per_sqm_by_district
train_room_means = room_means
train_life_square_means = life_square_means
train_square_means = square_means
train_district_means = df_processed_train.groupby('DistrictId')['Price'].mean()
train_overall_mean_price = df_processed_train['Price'].mean()

# Сохраняем средние для новых признаков взаимодействия
train_rooms_square_ratio_mean = df_processed_train['Rooms_Square_Ratio'].mean()
train_life_square_ratio_mean = df_processed_train['LifeSquare_Ratio'].mean()

# Создаем признаки
df_processed_train['DistrictId_TargetEnc'] = df_processed_train['DistrictId'].map(target_encoding)
df_processed_train['Avg_Price_Per_Sqm_By_District'] = df_processed_train['DistrictId'].map(price_per_sqm_by_district)

# Объединяем средние по комнатам
df_processed_train = df_processed_train.merge(room_means, on=['DistrictId', 'Rooms'], how='left')

# НОВЫЙ ПРИЗНАК: Объединяем средние по жилой площади
df_processed_train = df_processed_train.merge(life_square_means, on=['DistrictId', 'Rooms'], how='left')

# НОВЫЙ ПРИЗНАК: Объединяем средние по общей площади
df_processed_train = df_processed_train.merge(square_means, on=['DistrictId', 'Rooms'], how='left')

# Заполняем пропуски в новых признаках
overall_life_square_price = df_processed_train['Price'].mean() / df_processed_train['LifeSquare'].mean() if df_processed_train['LifeSquare'].mean() > 0 else 0
overall_square_price = df_processed_train['Price'].mean() / df_processed_train['Square'].mean() if df_processed_train['Square'].mean() > 0 else 0

if 'Mean_Price_By_LifeSquare' in df_processed_train.columns:
    df_processed_train['Mean_Price_By_LifeSquare'] = df_processed_train['Mean_Price_By_LifeSquare'].fillna(overall_life_square_price)
else:
    df_processed_train['Mean_Price_By_LifeSquare'] = overall_life_square_price

if 'Mean_Price_By_Square' in df_processed_train.columns:
    df_processed_train['Mean_Price_By_Square'] = df_processed_train['Mean_Price_By_Square'].fillna(overall_square_price)
else:
    df_processed_train['Mean_Price_By_Square'] = overall_square_price

# Заполняем пропуски в Mean_Price_By_Rooms
df_processed_train['Mean_Price_By_Rooms'] = df_processed_train['Mean_Price_By_Rooms'].fillna(
    df_processed_train['DistrictId'].map(train_district_means)
)

# Удаляем столбцы
columns_to_drop = ['Id', 'Healthcare_1','Ecology_1']
df_processed_train = df_processed_train.drop(columns=columns_to_drop)

# Подготовка категориальных признаков
categorical_columns = df_processed_train.select_dtypes(include=['object']).columns
for col in categorical_columns:
    df_processed_train[col] = df_processed_train[col].astype(str)

print("✅ ПРИЗНАКИ ДЛЯ ОБУЧЕНИЯ ПОДГОТОВЛЕНЫ")
print(f"Добавлены новые признаки:")
print(f"DistrictId_TargetEnc - средняя цена квартир в каждом районе (таргет-энкодинг по DistrictId)")
print(f"Mean_Price_By_Rooms - средняя цена квартир по комбинации 'район + количество комнат'")
print(f"Mean_Price_By_Square - средняя цена за кв.м общей площади по 'район + комнаты'")
print(f"Mean_Price_By_LifeSquare - средняя цена за кв.м ЖИЛОЙ площади по 'район + комнаты'")
print(f"Avg_Price_Per_Sqm_By_District - средняя цена за кв.м по районам (без учета комнат)")
print(f"LifeSquare_Ratio - доля жилой площади от общей (эффективность планировки)")
print(f"Rooms_Square_Ratio - плотность комнат (количество комнат на кв.метр)")


✅ ПРИЗНАКИ ДЛЯ ОБУЧЕНИЯ ПОДГОТОВЛЕНЫ
Добавлены новые признаки:
DistrictId_TargetEnc - средняя цена квартир в каждом районе (таргет-энкодинг по DistrictId)
Mean_Price_By_Rooms - средняя цена квартир по комбинации 'район + количество комнат'
Mean_Price_By_Square - средняя цена за кв.м общей площади по 'район + комнаты'
Mean_Price_By_LifeSquare - средняя цена за кв.м ЖИЛОЙ площади по 'район + комнаты'
Avg_Price_Per_Sqm_By_District - средняя цена за кв.м по районам (без учета комнат)
LifeSquare_Ratio - доля жилой площади от общей (эффективность планировки)
Rooms_Square_Ratio - плотность комнат (количество комнат на кв.метр)


C:\Users\Админ\AppData\Local\Temp\ipykernel_19084\2346112067.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  life_square_means = df_processed_train.groupby(['DistrictId', 'Rooms']).apply(
C:\Users\Админ\AppData\Local\Temp\ipykernel_19084\2346112067.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  square_means = df_processed_train.groupby(['DistrictId', 'Rooms']).apply(


In [22]:
# Определение фичей и таргета
target = 'Price'
features = [col for col in df_processed_train.columns if col != target]

# Определение категориальных фичей
cat_features = [i for i, col in enumerate(features) if df_processed_train[col].dtype == "object"]

# Разделение на train/valid
X_train, X_valid, y_train, y_valid = train_test_split(
    df_processed_train[features], df_processed_train[target],
    #test_size=0.2, random_state=42
)

# Создание Pool объектов
#train_pool = Pool(X_train, y_train, cat_features=cat_features)
#valid_pool = Pool(X_valid, y_valid, cat_features=cat_features)


In [23]:
# ====== Модель Gradient Boosting ======
model = GradientBoostingRegressor(
    n_estimators=600,          # количество слабых моделей (деревьев)
    learning_rate=0.03,        # скорость обучения
    max_depth=3,               # глубина каждого дерева
    subsample=0.8,             # стохастический бустинг, уменьшает переобучение
    random_state=42
)

In [24]:
model.fit(X_train, y_train)

GradientBoostingRegressor(learning_rate=0.03, n_estimators=600, random_state=42,
                          subsample=0.8)

In [26]:
# ====== Предсказания ======
preds = model.predict(X_valid)

In [30]:

# Метрики
print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ ОБУЧЕНИЯ:")
print("="*50)
print(f"MAE: {mean_absolute_error(y_valid, preds):.2f}")
print(f"R2: {r2_score(y_valid, preds):.4f}")
print("="*50)


РЕЗУЛЬТАТЫ ОБУЧЕНИЯ:
MAE: 27439.99
R2: 0.7625


In [31]:
# ====== Важность признаков ======
importance = pd.Series(model.feature_importances_, index=features)
print("\nВажность признаков:")
print(importance.sort_values(ascending=False))


Важность признаков:
Mean_Price_By_Rooms              0.672924
Square                           0.121801
Mean_Price_By_Square             0.046814
Rooms_Square_Ratio               0.044747
KitchenSquare                    0.018908
HouseYear                        0.016394
Mean_Price_By_LifeSquare         0.012677
HouseFloor                       0.009304
Floor                            0.009104
Avg_Price_Per_Sqm_By_District    0.007950
LifeSquare                       0.006954
Social_1                         0.006889
DistrictId_TargetEnc             0.005730
LifeSquare_Ratio                 0.004465
Social_2                         0.003898
Social_3                         0.003894
DistrictId                       0.003089
Shops_1                          0.001846
Helthcare_2                      0.001626
Rooms                            0.000562
Shops_2                          0.000192
Ecology_2                        0.000129
Ecology_3                        0.000102
dtype: float6

In [32]:
# Сравнение обученой модели с реальными ценами
comparison = pd.DataFrame({
    "Price_real": y_valid,
    "Price_pred": preds
})

comparison.head(20) 


,Price_real,Price_pred
7174,228956.20,206495.797124
3016,402216.53,239646.444981
7541,403657.90,392976.096067
7292,147017.77,151248.367534
2123,187421.70,130125.187502
4356,190686.66,191920.894155
1137,324328.06,328071.986704
4370,538574.75,412889.859275
3206,241284.70,232078.468742
6003,97026.19,124379.407381


In [33]:
df_processed_train.corr().style.background_gradient(cmap='coolwarm').format('{:.2f}')

,DistrictId,Rooms,Square,LifeSquare,KitchenSquare,Floor,HouseFloor,HouseYear,Ecology_2,Ecology_3,Social_1,Social_2,Social_3,Helthcare_2,Shops_1,Shops_2,Price,Rooms_Square_Ratio,LifeSquare_Ratio,DistrictId_TargetEnc,Avg_Price_Per_Sqm_By_District,Mean_Price_By_Rooms,Mean_Price_By_LifeSquare,Mean_Price_By_Square
DistrictId,1.00,0.07,-0.03,-0.09,0.04,-0.12,-0.15,-0.20,0.09,0.03,0.25,0.17,0.14,0.31,0.17,0.02,0.27,0.07,-0.10,0.45,0.50,0.33,0.47,0.48
Rooms,0.07,1.00,0.66,0.51,0.01,-0.00,-0.03,-0.04,0.00,0.01,0.08,0.07,0.01,0.06,0.05,0.00,0.55,0.39,0.06,0.15,0.11,0.69,-0.13,-0.04
Square,-0.03,0.66,1.00,0.77,0.01,0.11,0.08,0.18,-0.03,-0.02,-0.07,-0.04,0.04,-0.02,0.02,0.05,0.52,-0.03,0.03,0.05,-0.06,0.48,-0.25,-0.18
LifeSquare,-0.09,0.51,0.77,1.00,-0.01,0.11,0.08,0.13,-0.03,-0.04,-0.19,-0.16,0.07,-0.10,-0.00,0.05,0.32,0.00,0.46,-0.07,-0.17,0.30,-0.38,-0.26
KitchenSquare,0.04,0.01,0.01,-0.01,1.00,-0.01,0.00,0.04,-0.00,0.01,0.04,0.04,-0.02,0.04,0.01,0.02,0.03,-0.01,-0.03,0.04,0.04,0.03,0.05,0.04
Floor,-0.12,-0.00,0.11,0.11,-0.01,1.00,0.42,0.28,-0.05,-0.03,-0.04,-0.02,-0.00,-0.07,0.02,0.01,0.13,-0.08,0.04,-0.01,-0.07,0.01,-0.09,-0.07
HouseFloor,-0.15,-0.03,0.08,0.08,0.00,0.42,1.00,0.39,-0.06,-0.01,-0.02,0.01,-0.01,-0.07,0.03,-0.05,0.09,-0.11,0.03,-0.04,-0.08,-0.02,-0.08,-0.07
HouseYear,-0.20,-0.04,0.18,0.13,0.04,0.28,0.39,1.00,-0.08,-0.05,-0.05,0.01,-0.05,-0.12,-0.00,0.06,0.04,-0.16,0.02,-0.14,-0.23,-0.09,-0.20,-0.22
Ecology_2,0.09,0.00,-0.03,-0.03,-0.00,-0.05,-0.06,-0.08,1.00,-0.02,0.07,0.01,-0.01,0.08,-0.05,0.02,-0.02,0.02,-0.01,-0.03,0.02,-0.02,0.03,0.02
Ecology_3,0.03,0.01,-0.02,-0.04,0.01,-0.03,-0.01,-0.05,-0.02,1.00,0.04,-0.01,-0.01,0.13,-0.00,-0.05,0.05,0.02,-0.03,0.09,0.11,0.06,0.12,0.11
